In [11]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
from sklearn.linear_model import LogisticRegression as lr

In [4]:
xtrain = pd.read_csv("X_train.csv")
xtrain = xtrain.iloc[:,2:] # Remove CustomerId and Surname
xtrain.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,791,Germany,Female,35,7,52436.20,1,1,0,161051.75
1,705,Germany,Male,42,8,166685.92,2,1,1,55313.51
2,543,France,Female,31,4,138317.94,1,0,0,61843.73
3,709,France,Female,32,2,0.00,2,0,0,109681.29
4,714,Germany,Female,36,1,101609.01,2,1,1,447.73


In [6]:
def preprocess_data(df):
    # 1️⃣ Drop first two columns (CustomerId, Surname)
    df = df.iloc[:, 2:].copy()

    # 2️⃣ Clean and encode Gender
    if "Gender" in df.columns:
        df["Gender"] = df["Gender"].astype(str).str.strip().str.lower()
        df["Gender"] = df["Gender"].map({"female": 0, "male": 1})
        df["Gender"] = df["Gender"].fillna(df["Gender"].median())

    # 3️⃣ Encode Geography as categorical codes
    if "Geography" in df.columns:
        df["Geography"] = df["Geography"].astype("category").cat.codes

    # 4️⃣ Select important features
    selected_features = ["Age", "Balance", "Geography", "EstimatedSalary"]
    df = df[selected_features]

    return df

# Strip whitespace and lowercase first
xtrain["Gender"] = xtrain["Gender"].str.strip().str.lower()
xtrain["Gender"] = xtrain["Gender"].map({"female": 0, "male": 1})
xtrain["Geography"] = xtrain["Geography"].astype("category").cat.codes
xtrain["Gender"] = xtrain["Gender"].fillna(xtrain["Gender"].median())
xtrain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6499 entries, 0 to 6498
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      6499 non-null   int64  
 1   Geography        6499 non-null   int8   
 2   Gender           6499 non-null   int64  
 3   Age              6499 non-null   int64  
 4   Tenure           6499 non-null   int64  
 5   Balance          6499 non-null   float64
 6   NumOfProducts    6499 non-null   int64  
 7   HasCrCard        6499 non-null   int64  
 8   IsActiveMember   6499 non-null   int64  
 9   EstimatedSalary  6499 non-null   float64
dtypes: float64(2), int64(7), int8(1)
memory usage: 463.4 KB


In [8]:
selected_features  = ["Age", "Balance", "Geography", "EstimatedSalary"]
xtrain = xtrain[selected_features]
display(xtrain.head())
print(xtrain.shape)

,Age,Balance,Geography,EstimatedSalary
0,35,52436.20,1,161051.75
1,42,166685.92,1,55313.51
2,31,138317.94,0,61843.73
3,32,0.00,0,109681.29
4,36,101609.01,1,447.73


(6499, 4)


In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(xtrain.values)
ytrain = pd.read_csv("y_train.csv")
ytrain = ytrain.iloc[:,-1]

In [14]:
model = lr().fit(X_train_scaled,ytrain.values)

In [15]:
xtest = preprocess_data(pd.read_csv("X_test.csv"))
xtest.head()

,Age,Balance,Geography,EstimatedSalary
0,60,92887.06,0,39473.63
1,56,115895.22,0,4176.17
2,32,75170.54,0,37898.50
3,34,107556.06,2,154631.35
4,62,0.00,0,100941.57


In [18]:
X_test = xtest.values
X_test_scaled = scaler.transform(X_test)
ypred = model.predict(X_test_scaled)

In [19]:
ytest = pd.read_csv("y_test.csv").iloc[:, -1]

print("Accuracy:", accuracy_score(ytest, ypred))
print("Precision:", precision_score(ytest, ypred))
print("Recall:", recall_score(ytest, ypred))
print("F1 Score:", f1_score(ytest, ypred))
print("\nConfusion Matrix:\n", confusion_matrix(ytest, ypred))
print("\nClassification Report:\n", classification_report(ytest, ypred))

Accuracy: 0.7777777777777778
Precision: 0.2847682119205298
Recall: 0.06030855539971949
F1 Score: 0.09953703703703703

Confusion Matrix:
 [[2680  108]
 [ 670   43]]

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.96      0.87      2788
           1       0.28      0.06      0.10       713

    accuracy                           0.78      3501
   macro avg       0.54      0.51      0.49      3501
weighted avg       0.70      0.78      0.72      3501

